In [1]:
import os
import pandas as pd
import pickle

from textwave.modules.retrieval.index.bruteforce import FaissBruteForce
from textwave.modules.retrieval.search import FaissSearch

from textwave.modules.generator.question_answering import QAGeneratorMistral
from textwave.modules.retrieval.reranker import Reranker
from textwave.modules.utils.metrics import Matching

print('DONE')

DONE


In [28]:
QUESTIONS_PATH = 'textwave/qa_resources/question.tsv'
CORPUS_PATH = 'textwave/storage/'
CHUNKING_STRATEGY = 'fixed-length' # 'fixed-length' or 'sentence'
CHUNKING_PARAMETERS = {
    "chunk_size": 150, 
    "overlap_size": 0
}
INDEX_STRATEGY = "bruteforce"
INDEX_PARAMETERS = {
    'metric': 'cosine',
}
# K_NEAREST_NEIGHBORS = 3
MISTRAL_MODEL = 'mistral-medium-latest'
API_KEY = os.environ["MISTRAL_API_KEY"]

# 'm' context chunks...
M = 100

In [29]:
# Process questions df
raw_questions = pd.read_table(QUESTIONS_PATH)
easy = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'easy'].reset_index(drop=True)[:20]
medium = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'medium'].reset_index(drop=True)[:20]
hard = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'hard'].reset_index(drop=True)[:20]
questions = pd.concat([easy, medium, hard]).reset_index(drop=True)

In [5]:
# Use best performing index
index = FaissBruteForce.load('faiss/bruteforce_cosine_fixed-length_150_0.pkl')

In [6]:
# Load question embeddings
with open('analysis/question_embeddings.pkl', 'rb') as f:
    embeddings_map = pickle.load(f)

In [7]:
from sentence_transformers import CrossEncoder
model = CrossEncoder(
    "zli12321/answer_equivalence_distilbert",
    num_labels=2
)

In [30]:
# Initialize objects
mistral = QAGeneratorMistral(API_KEY, generator_model=MISTRAL_MODEL)
search = FaissSearch(index, metric=INDEX_PARAMETERS['metric'])
rerank = Reranker(type='tfidf')

# Get unique questions 
unique_questions = questions['Question'].unique()
results = {}
count = 0
for question in unique_questions:
    print(f'{count+1}/{len(unique_questions)}')

    # Embed query
    query_vector = embeddings_map[question]

    # Search index for neighbors
    _, _, meta_results = search.search(query_vector, k=M)
    print(len(meta_results))

    # Perform re-ranking - choose "tfidf re-rank" for this
    reranked_context, _, _ = rerank.rerank(question, meta_results)

    # Trigger QA object to ping MISTRAL, get reponse, return
    answer = mistral.generate_answer(query=question, context=reranked_context)
    results[question] = answer

    # Increment count
    count += 1

results

1/41
100
2/41
100
3/41
100
4/41
100
5/41
100
6/41
100
7/41
100
8/41
100
9/41
100
10/41
100
11/41
100
12/41
100
13/41
100
14/41
100
15/41
100
16/41
100
17/41
100
18/41
100
19/41
100
20/41
100
21/41
100
22/41
100
23/41
100
24/41
100
25/41
100
26/41
100
27/41
100
28/41
100
29/41
100
30/41
100
31/41
100
32/41
100
33/41
100
34/41
100
35/41
100
36/41
100
37/41
100
38/41
100
39/41
100
40/41
100
41/41
100


{'Was Abraham Lincoln the sixteenth President of the United States?': 'Yes, Abraham Lincoln was the sixteenth President of the United States. The context confirms that he served from **March 4, 1861**, until his assassination in **1865**, and was elected in **1860** as the 16th president.',
 'Did Lincoln sign the National Banking Act of 1863?': 'No context.',
 'Did his mother die of pneumonia?': 'No context.',
 "How many long was Lincoln's formal education?": "Abraham Lincoln's formal education consisted of about **18 months of schooling**.",
 'When did Lincoln begin his political career?': 'Abraham Lincoln began his political career in **1832**, at the age of **23**.',
 'What did The Legal Tender Act of 1862 establish?': 'The **Legal Tender Act of 1862** established the **United States Note**, the first **paper currency** in the U.S., which served as legal tender to help finance the Civil War and address a currency crisis.',
 'Was Abraham Lincoln the first President of the United Stat

In [31]:
# Load pickle file
with open('results.pkl', 'rb') as file:
    results = pickle.load(file)

# Get metrcis, add to dataframe
metrics = Matching(model='cross-encoder/nli-distilroberta-base')

processed = questions[~questions['Question'].isna()]
processed = processed[~processed['Answer'].isna()]
indices = []
for idx, row in processed.iterrows():
    if row['Question'] not in results:
        indices.append(idx)
processed = processed.drop(indices)

for idx, row in processed.iterrows():
    question = row['Question']
    print(f'{idx+1}/{len(processed)+1}')

    true_answer = row['Answer']
    generated_answer = results[question]

    em = metrics.exact_match(generated_answer, true_answer)
    print(f"Exact Match: {em}")

    processed.at[idx, 'Exact Match'] = em

1/60
Exact Match: True
2/60
Exact Match: True
3/60
Exact Match: False
4/60
Exact Match: True
5/60
Exact Match: True
6/60
Exact Match: True
7/60
Exact Match: False
8/60
Exact Match: True
9/60
Exact Match: True
10/60
Exact Match: True
11/60
Exact Match: False
12/60
Exact Match: False
13/60
Exact Match: True
14/60
Exact Match: True
15/60
Exact Match: True
16/60
Exact Match: False
17/60
Exact Match: False
18/60
Exact Match: True
19/60
Exact Match: False
20/60
Exact Match: False
21/60
Exact Match: False
22/60
Exact Match: True
23/60
Exact Match: True
24/60
Exact Match: True
25/60
Exact Match: False
26/60
Exact Match: False
27/60
Exact Match: False
28/60
Exact Match: True
29/60
Exact Match: True
30/60
Exact Match: False
31/60
Exact Match: False
32/60
Exact Match: True
33/60
Exact Match: False
34/60
Exact Match: False
35/60
Exact Match: False
36/60
Exact Match: True
37/60
Exact Match: False
38/60
Exact Match: True
39/60
Exact Match: False
40/60
Exact Match: True
41/60
Exact Match: False
42/60

In [32]:
n = processed[~processed['Exact Match'].isna()]

print(len(n[n['Exact Match']==True]) / len(n))

# easy = n[n['DifficultyFromQuestioner'] == 'easy']
# medium = n[n['DifficultyFromQuestioner'] == 'medium']
# hard = n[n['DifficultyFromQuestioner'] == 'hard']

# dfs = [easy, medium, hard]

# for df in dfs:
#     print(len(df[df['Exact Match']==True]) / len(df))
#     # print(len(df[df['Transformer Match']==True]) / len(df))

0.4915254237288136
